# OwnGPT : entraîner ton propre GPT français, de zéro, sur un GPU gratuit

Quatre étapes, toutes reprenables (une coupure fait perdre au plus 250 steps) :
1. **Tokenizer** : ton propre vocabulaire BPE, appris sur du français.
2. **Préentraînement** : FineWeb2-HQ (web français filtré) + Wikipédia FR. Le modèle apprend la langue.
3. **Chat (SFT)** : French-Alpaca + conversations traduites + l'identité d'OwnGPT. Il apprend à répondre.
4. **Test et export** du modèle final.

**Où le lancer (gratuit)**
- **Kaggle (recommandé)** : 30 h de GPU par semaine, sessions de 12 h. New Notebook > File > Import Notebook,
  puis Settings : Accelerator `GPU T4 x2` (un seul est utilisé), Internet `On` (vérification par téléphone requise).
  Pour que ça tourne navigateur fermé : **Save Version > Save & Run All**.
- **Colab** : Exécution > Modifier le type d'exécution > GPU T4. Sessions plus courtes, tout est gardé sur Google Drive.

**Après une coupure** : relance tout, l'entraînement reprend où il s'était arrêté.
Sur Kaggle en mode *Save Version*, ajoute d'abord la sortie de la version précédente en *Input*
(Add Input > Your Work) : le notebook la récupère tout seul.

In [ ]:
import os, sys, glob, shutil, time
T0 = time.time()                                 # session start: training budgets count from here
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    STORE, WORK = '/content/drive/MyDrive/owngpt', '/content'
elif IN_KAGGLE:
    STORE, WORK = '/kaggle/working/owngpt', '/kaggle/working'
    previous = sorted(glob.glob('/kaggle/input/*/owngpt'))
    if previous and not os.path.exists(STORE):   # resume from an earlier saved version
        shutil.copytree(previous[-1], STORE)
        print('repris depuis', previous[-1])
else:
    STORE, WORK = os.path.abspath('owngpt_store'), os.getcwd()
os.makedirs(STORE, exist_ok=True)
print('stockage :', STORE)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd {WORK}
!test -d OwnAI || git clone -q https://github.com/Zlarien/OwnAI.git
%cd {WORK}/OwnAI
!git pull -q
!pip install -q -e ".[gpt]"

## Réglages
Calibrés pour un T4 (16 Go). Un modèle apprend bien avec au moins ~20 tokens par paramètre,
soit ~0,85 milliard pour le preset `t4` (42M paramètres). La vitesse réelle (tok/s) et le temps
restant s'affichent pendant l'entraînement.

In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
BIG = any(k in gpu for k in ('A100', 'H100', 'L4'))
PRESET = 'base' if BIG else 't4'
BLOCK = 1024 if BIG else 512
TOKENS = 3e9 if BIG else 1e9
BATCH, ACCUM = (16, 32) if BIG else (32, 4)          # ~524k / ~65k tokens par step
STEPS = int(TOKENS // (BATCH * ACCUM * BLOCK))
SESSION = 11.0 if IN_KAGGLE else 3.5                 # Kaggle coupe à 12 h : on garde une marge

def hours_left():
    # Time still available in this session, so training stops and saves before the cut.
    return round(max(0.25, SESSION - (time.time() - T0) / 3600), 2)
DATA = '/kaggle/temp/data' if IN_KAGGLE else f'{WORK}/data'   # copie de travail locale, hors sauvegarde
TOK = f'{STORE}/tokenizer.json'
print(gpu, PRESET, f'{TOKENS:.0e} tokens', STEPS, 'steps')

def sync(name):
    # Prepared data lives on the persistent store; training reads a local copy.
    src, dst = f'{STORE}/data/{name}', f'{DATA}/{name}'
    if os.path.exists(f'{dst}/meta.json'):
        return True
    if os.path.exists(f'{src}/meta.json'):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        return True
    return False

def save_data(name):
    shutil.copytree(f'{DATA}/{name}', f'{STORE}/data/{name}', dirs_exist_ok=True)

## 1. Ton tokenizer (quelques minutes, une seule fois)

In [ ]:
if not os.path.exists(TOK):
    !python -m ownai.gpt train-tokenizer --out {TOK} --chars 2e8
from ownai.gpt.tokenizer import Tokenizer
t = Tokenizer.load(TOK)
print(t.n_vocab, [t.decode([i]) for i in t.encode("L'Assemblée nationale a adopté la réforme.")])

## 2. Données de préentraînement (téléchargées et tokenisées une fois)

In [ ]:
if not sync('pretrain'):
    !python -m ownai.gpt prepare-pretrain --out {DATA}/pretrain --tokenizer {TOK} --tokens {TOKENS}
    save_data('pretrain')
!cat {DATA}/pretrain/meta.json

## 3. Préentraînement (relance cette cellule après une coupure : elle reprend)

In [ ]:
args = (f'--stage pretrain --preset {PRESET} --data {DATA}/pretrain --out {STORE}/runs/pretrain '
        f'--max-steps {STEPS} --batch-size {BATCH} --grad-accum {ACCUM} --eval-every 250 --save-every 250 '
        f'--hours {hours_left()}')
# torch.compile speeds training up when the GPU supports it; if it fails, rerun plainly (it resumes).
!python -m ownai.gpt train {args} --compile || python -m ownai.gpt train {args}

In [ ]:
import json
import matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f'{STORE}/runs/pretrain/log.jsonl', encoding='utf-8')]
tr = [(r['step'], r['loss']) for r in rows if 'loss' in r]
va = [(r['step'], r['val_loss']) for r in rows if 'val_loss' in r]
if tr: plt.plot(*zip(*tr), label='train', alpha=.5)
if va: plt.plot(*zip(*va), label='val')
plt.legend(); plt.xlabel('step'); plt.ylabel('loss'); plt.show()
for r in [r for r in rows if 'sample' in r][-5:]:
    print(r['step'], f"bpb {r['val_bpb']:.3f}", '|', r['sample'][:200])

## 4. Données de chat

In [ ]:
if not sync('sft'):
    !python -m ownai.gpt prepare-sft --out {DATA}/sft --tokenizer {TOK}
    save_data('sft')
!cat {DATA}/sft/meta.json

## 5. Entraînement au chat (a besoin de `runs/pretrain/model.pt`, écrit quand le préentraînement est fini)

In [ ]:
if os.path.exists(f'{STORE}/runs/pretrain/model.pt'):
    args = (f'--stage sft --init {STORE}/runs/pretrain/model.pt --data {DATA}/sft --out {STORE}/runs/sft '
            f'--max-steps 3000 --batch-size {BATCH} --grad-accum 4 --eval-every 250 --save-every 250 '
            f'--hours {hours_left()}')
    !python -m ownai.gpt train {args} --compile || python -m ownai.gpt train {args}
else:
    print('Préentraînement pas encore fini : relance une nouvelle session pour continuer.')

## 6. Parle-lui

In [ ]:
from ownai.gpt.chat import OwnGPT
final = f'{STORE}/runs/sft/model.pt'
if os.path.exists(final):
    bot = OwnGPT.load(final)
    for q in ['Qui es-tu ?', 'Quelle est la capitale de la France ?',
              "Explique ce qu'est un réseau de neurones en deux phrases.",
              'Donne-moi trois conseils pour apprendre à coder.']:
        print('Q :', q)
        print('R :', bot.reply([{'role': 'user', 'content': q}]), '\n')
else:
    print("Pas encore de modèle de chat : il apparaîtra quand l'étape 5 sera finie.")

## 7. Récupérer le modèle
Télécharge `owngpt/runs/sft/model.pt` (Colab : Google Drive, Kaggle : onglet Output),
place-le dans le repo à `artifacts/owngpt/model.pt`, puis lance `python -m ownai.cli serve` :
le mode **OwnGPT** apparaît dans la page, avec les réponses en direct.